# The Power Method for Finding Eigenvalues

## Setup

Given a square matrix $A \in \mathbb{R}^{n \times n}$, we want to find its **dominant eigenvalue** $\lambda_1$ — the one with the largest absolute value — and its corresponding eigenvector $\mathbf{v}_1$.

We assume $A$ has $n$ linearly independent eigenvectors $\mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_n$ with eigenvalues satisfying:

$$|\lambda_1| > |\lambda_2| \geq |\lambda_3| \geq \cdots \geq |\lambda_n|$$

---

## Key Idea: Repeated Multiplication

Start with any random vector $\mathbf{b}_0$ (that isn't orthogonal to $\mathbf{v}_1$). We can express it in the eigenvector basis:

$$\mathbf{b}_0 = c_1\mathbf{v}_1 + c_2\mathbf{v}_2 + \cdots + c_n\mathbf{v}_n, \quad c_1 \neq 0$$

Now apply $A$ repeatedly:

$$A^k \mathbf{b}_0 = c_1 \lambda_1^k \mathbf{v}_1 + c_2 \lambda_2^k \mathbf{v}_2 + \cdots + c_n \lambda_n^k \mathbf{v}_n$$

Factor out $\lambda_1^k$:

$$A^k \mathbf{b}_0 = \lambda_1^k \left[ c_1 \mathbf{v}_1 + c_2\left(\frac{\lambda_2}{\lambda_1}\right)^k \mathbf{v}_2 + \cdots + c_n\left(\frac{\lambda_n}{\lambda_1}\right)^k \mathbf{v}_n \right]$$

Since $\left|\dfrac{\lambda_i}{\lambda_1}\right| < 1$ for all $i > 1$, every term except the first **vanishes** as $k \to \infty$:

$$\frac{A^k \mathbf{b}_0}{\|A^k \mathbf{b}_0\|} \;\longrightarrow\; \mathbf{v}_1 \quad \text{as } k \to \infty$$

---

## The Algorithm

**Initialize:** Choose a random unit vector $\mathbf{b}_0$ with $\|\mathbf{b}_0\| = 1$.

**Iterate** for $k = 0, 1, 2, \dots$:

$$\mathbf{w} = A\,\mathbf{b}_k$$

$$\mathbf{b}_{k+1} = \frac{\mathbf{w}}{\|\mathbf{w}\|}$$

**Estimate the eigenvalue** via the Rayleigh quotient:

$$\lambda_k = \mathbf{b}_k^\top A\,\mathbf{b}_k = \frac{\mathbf{b}_k^\top A\,\mathbf{b}_k}{\mathbf{b}_k^\top \mathbf{b}_k}$$

**Stop** when the estimate converges:

$$|\lambda_k - \lambda_{k-1}| < \varepsilon$$

---

## Why the Rayleigh Quotient?

For any unit vector $\mathbf{b}$, the Rayleigh quotient

$$R(\mathbf{b}) = \mathbf{b}^\top A\,\mathbf{b}$$

gives the best scalar approximation to the eigenvalue associated with the direction of $\mathbf{b}$. Once $\mathbf{b}_k \approx \mathbf{v}_1$, we get $R(\mathbf{b}_k) \approx \lambda_1$ because:

$$\mathbf{b}_k^\top A\,\mathbf{b}_k \;\approx\; \mathbf{v}_1^\top A\,\mathbf{v}_1 = \mathbf{v}_1^\top (\lambda_1 \mathbf{v}_1) = \lambda_1 \|\mathbf{v}_1\|^2 = \lambda_1$$

---

## Convergence Rate

The error in the eigenvector estimate shrinks geometrically each iteration:

$$\text{error} \sim \left|\frac{\lambda_2}{\lambda_1}\right|^k$$

Convergence is **fast** when $|\lambda_2 / \lambda_1| \ll 1$ (eigenvalues well-separated), and **slow** when the two largest eigenvalues are close in magnitude.

---

## Limitations

| Concern | Detail |
|---|---|
| Finds only $\lambda_1$ | Must deflate $A$ to find subsequent eigenvalues |
| Requires $\vert\lambda_1\vert > \vert\lambda_2\vert$ | Fails if two eigenvalues tie in magnitude |
| Starting vector | Must have $c_1 \neq 0$ (virtually guaranteed randomly) |
| Complex eigenvalues | Real power method assumes real dominant eigenvalue |

In [1]:
import numpy as np 

In [2]:
# Finding Eigenvalues via Power Iteration
def power_iteration(A, num_simulations: int):
    # Step 1: Start with a random vector
    b_k = np.random.rand(A.shape[1])

    for _ in range(num_simulations):
        # Step 2: Calculate the matrix-by-vector product Ab
        b_k1 = A @ b_k

        # Step 3: Normalize the resulting vector
        b_k1_norm = np.linalg.norm(b_k1)
        b_k = b_k1 / b_k1_norm

    # Step 4: Rayleigh quotient to estimate the eigenvalue
    eigenvalue = b_k.T @ (A @ b_k) / (b_k.T @ b_k)

    return eigenvalue, b_k

# Newton's Method for Nonlinear Systems via the Jacobian

## The Problem

We want to find $(x^*, y^*)$ such that both equations are simultaneously satisfied:

$$f(x, y) = 0, \qquad g(x, y) = 0$$

We write this compactly as a vector equation:

$$\mathbf{F}(\mathbf{x}) = \mathbf{0}, \qquad \mathbf{F} = \begin{bmatrix} f(x,y) \\ g(x,y) \end{bmatrix}, \quad \mathbf{x} = \begin{bmatrix} x \\ y \end{bmatrix}$$

---

## The Key Idea: Linearise Around the Current Guess

Suppose we have a current guess $\mathbf{x}_k$. Taylor-expand $\mathbf{F}$ to first order around it:

$$\mathbf{F}(\mathbf{x}_k + \boldsymbol{\delta}) \approx \mathbf{F}(\mathbf{x}_k) + J(\mathbf{x}_k)\,\boldsymbol{\delta}$$

where $J$ is the **Jacobian matrix** of partial derivatives evaluated at $\mathbf{x}_k$:

$$J(\mathbf{x}_k) = \begin{bmatrix} \dfrac{\partial f}{\partial x} & \dfrac{\partial f}{\partial y} \\[10pt] \dfrac{\partial g}{\partial x} & \dfrac{\partial g}{\partial y} \end{bmatrix}_{\mathbf{x}_k}$$

We want the update $\boldsymbol{\delta}$ to make $\mathbf{F}(\mathbf{x}_k + \boldsymbol{\delta}) = \mathbf{0}$, so we set:

$$\mathbf{F}(\mathbf{x}_k) + J(\mathbf{x}_k)\,\boldsymbol{\delta} = \mathbf{0}$$

$$\boxed{J(\mathbf{x}_k)\,\boldsymbol{\delta} = -\mathbf{F}(\mathbf{x}_k)}$$

This is just a $2 \times 2$ **linear system** — solve it for $\boldsymbol{\delta}$, then step forward.

---

## The Algorithm

**Initialize:** Choose an initial guess $\mathbf{x}_0 = (x_0, y_0)^T$.

**Iterate** for $k = 0, 1, 2, \dots$:

**1.** Evaluate the residual vector:
$$\mathbf{F}(\mathbf{x}_k) = \begin{bmatrix} f(x_k, y_k) \\ g(x_k, y_k) \end{bmatrix}$$

**2.** Build the Jacobian at the current point:
$$J_k = \begin{bmatrix} f_x(x_k, y_k) & f_y(x_k, y_k) \\ g_x(x_k, y_k) & g_y(x_k, y_k) \end{bmatrix}$$

**3.** Solve the linear system for the Newton step $\boldsymbol{\delta}_k$:
$$J_k\,\boldsymbol{\delta}_k = -\mathbf{F}(\mathbf{x}_k)$$

In closed form (via Cramer's rule or direct inversion when $2\times2$):
$$\boldsymbol{\delta}_k = -J_k^{-1}\,\mathbf{F}(\mathbf{x}_k), \qquad \det(J_k) \neq 0$$

**4.** Update the iterate:
$$\mathbf{x}_{k+1} = \mathbf{x}_k + \boldsymbol{\delta}_k$$

**Stop** when sufficiently converged:
$$\|\mathbf{F}(\mathbf{x}_{k+1})\| < \varepsilon \quad \text{or} \quad \|\boldsymbol{\delta}_k\| < \varepsilon$$

---

## Why Does It Work? (The Geometry)

At each step, Newton's method replaces the nonlinear system with its **tangent plane approximation** — a flat linearisation — and finds where that plane crosses zero. In 1D this is classic Newton's root-finding:

$$x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}$$

The Jacobian $J$ is exactly the multi-dimensional analogue of $f'$: it captures all first-order rates of change between inputs and outputs. Dividing by $f'$ becomes multiplying by $J^{-1}$.

---

## Explicit $2\times 2$ Step

For our two-equation system, the Jacobian inverse is analytic. Let:

$$\Delta = \det(J_k) = f_x g_y - f_y g_x$$

Then:

$$J_k^{-1} = \frac{1}{\Delta} \begin{bmatrix} g_y & -f_y \\ -g_x & f_x \end{bmatrix}$$

So the Newton step is:

$$\begin{bmatrix} \delta_x \\ \delta_y \end{bmatrix} = \frac{-1}{\Delta} \begin{bmatrix} g_y\, f_k - f_y\, g_k \\ f_x\, g_k - g_x\, f_k \end{bmatrix}$$

where $f_k = f(x_k, y_k)$ and $g_k = g(x_k, y_k)$.

---

In [4]:
def multivariate_newton_method(func, jacobian, x0, tol=1e-6, max_iter=100):
    """
    Multivariate Newton's method for finding roots of a vector-valued function.

    Parameters:
    func: A function that takes a vector and returns a vector (the function whose root we want to find).
    jacobian: A function that takes a vector and returns the Jacobian matrix of the function.
    x0: Initial guess for the root.
    tol: Tolerance for convergence.
    max_iter: Maximum number of iterations.

    Returns:
    A tuple containing the estimated root and a boolean indicating whether convergence was achieved.
    """
    x = x0
    for i in range(max_iter):
        f_x = func(x)
        J_x = jacobian(x)

        # Solve J_x * delta = -f_x for delta
        try:
            delta = np.linalg.solve(J_x, -f_x)
        except np.linalg.LinAlgError:
            print("Jacobian is singular. Cannot proceed with Newton's method.")
            return x, False

        x = x + delta

        if np.linalg.norm(delta) < tol:
            return x, True

    print("Maximum iterations reached without convergence.")
    return x, False